In [ ]:
import torch.nn as nn 
import torch

GPT_CONFIG_124M = {
 "vocab_size": 50257, # Vocabulary size
 "context_length": 1024, # Context length
 "emb_dim": 768, # Embedding dimension
 "n_heads": 12, # Number of attention heads
 "n_layers": 12, # Number of layers
 "drop_rate": 0.1, # Dropout rate
 "qkv_bias": False # Query-Key-Value bias
}

# Everything within the self attention implement + tranformer blocks, final layer normalization, and output layer 

class DummyGPTModel(nn.Module):
    def __init__(self, cfg):
      super().__init__()
      self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
      self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
      self.drop_emb = nn.Dropout(cfg["drop_rate"])
      self.trf_blocks = nn.Sequential(
          *[DummyTransformerBlock(cfg) # Placeholder for Transformer Block  
          for _ in range(cfg["n_layers"])]
      )
      self.final_norm = DummyLayerNorm(cfg["emb_dim"]) # Placeholder for Layer Normalization 
      self.out_head = nn.Linear(
        cfg["emb_dim"], cfg["vocab_size"], bias=False
      )

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(
            torch.arange(seq_len, device=in_idx.device)
        )
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

    

In [ ]:
# Pseudo Implementation of Transformer Block & Layer Normalization 
class DummyTransformerBlock(nn.Module):
    def __init__(self, cfg):
      super().__init__()
      def forward(self, x):
        return x

# In modern LLM, layer normalization is applied before and after the multi-head attention module
class DummyLayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5):
        super().__init__()
    def forward(self, x):
        return x

    
# Real Multi-Head Attention 
class MultiHeadAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_heads = cfg["n_heads"]
        self.d_out = cfg["emb_dim"]
        self.head_dim = self.d_out // self.n_heads
        

# Real Layer Normalization
class LayerNorm(nn.Module): 
    def __init__(self, emb_dim): 
        super().__init__()
        self.eps = 1e-5 # Small constant to avoid division by zero
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        x_norm = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * x_norm + self.shift
    

# Layer Normalization 
## Normalization 1, Before Attention 

Layer normalization happens twice in a GPT3 and other Modern LLM
Suppose we have a token with the embedded vector of 4 values or a embedded vector of 4 dimensions. 

x = [2, 4, 6, 8]

Layer normalization normalizes through the input features. But we start off by finding the mean, variance, and standard deviations since those are the variable requried to compute layer normalization 

Formula = (value - mean) / std
normalized x ≈ [-1.34, -0.45, 0.45, 1.34]

Now this normalized vector goes into multi-head attention.

Q = norm1(x) @ Wq
K = norm1(x) @ Wk
V = norm1(x) @ Wv

## Normalization 2

After passing the normalized inputs into the the attention module, it outputs a context vector.
To be precise, learned context (weights) matrix multiplied by value vector matrix is the context vector.
We then take the attention output / context vector and multiply it with the original, unnormalized input matrix. 

Residual Addition 1:
original x = [2.0, 4.0, 6.0, 8.0]
attention_output = [0.5, -0.2, 1.0, 0.3]

x = [2.5, 3.8, 7.0, 8.3]

After residual attention, we then apply the second normalization which is before feed forward layer 
the process of normalization is the same as before. After getting the normalized input, we feed it into the feed-forward layer

ff_output = FeedForward(norm2(x))

After getting the feed forward output, you perform residual connection again. 

previous x = [2.5, 3.8, 7.0, 8.3]
ff_output  = [0.1, 0.7, -0.4, 0.2]

This returns the final output of the transformer block: 
x = [2.6, 4.5, 6.6, 8.5]


